<a href="https://colab.research.google.com/github/sargondzn/agent_asoif/blob/main/ai_agent_asoif.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Requirements

- langchain

- langchain-ollama

- requests

- pydantic

In [4]:
!apt-get -qq install -y zstd

Selecting previously unselected package zstd.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [5]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [6]:
import subprocess, time
subprocess.Popen(["ollama", "serve"],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

In [7]:
!ollama pull llama3.1:8b

In [24]:
!pip install -q langchain langchain-ollama requests
!pip install -q langgraph

In [35]:
from pydantic import BaseModel
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langgraph.prebuilt import create_react_agent
from langchain.tools import tool
import requests

In [ ]:
# There are three types of models we can call from Ollama:

# Chat Ollama = useful in our scenario. It can use tools to do its actions, exactly what an agent need.
# OllamaLLM = takes a string and returns a string, no mechanisms for tool calling.
# OllamaEmbeddings = good for RAG applications.

In [26]:
llm = ChatOllama(model="llama3.1:8b", temperature=0)

In [27]:
# Testing the server
print(llm.invoke("Say hello in five words.").content)

Hello, how are you today?


In [42]:
API = "https://awoiaf.westeros.org/api.php"
HEADERS = {"User-Agent": "asoif-study-bot/0.1"}
MAX_CHARS = 4000

In [ ]:
@tool
def search_wiki(query: str) -> str:
    """Search A Wiki of Ice and Fire for a topic and return the page text.

    Use this for any question about characters, houses, places, or events in
    A Song of Ice and Fire. Pass a short search term, ideally a page title
    such as "Jon Snow" or "House Stark".
    """
    try:
        r = requests.get(API, params={
            "action": "query",
            "list": "search",
            "srsearch": query,
            "format": "json",
        }, headers=HEADERS, timeout=20)
        r.raise_for_status()
        hits = r.json()["query"]["search"]

        if not hits:
            return f"No wiki page found for '{query}'."

        title = hits[0]["title"]

        r = requests.get(API, params={
            "action": "query",
            "prop": "extracts",
            "explaintext": 1,
            "titles": title,
            "format": "json",
        }, headers=HEADERS, timeout=20)
        r.raise_for_status()
        pages = r.json()["query"]["pages"]
        text = next(iter(pages.values())).get("extract", "")

        if not text.strip():
            return f"Page '{title}' exists but returned no text."

        if len(text) > MAX_CHARS:
            text = text[:MAX_CHARS] + "\n\n[truncated]"

        return f"Page: {title}\n\n{text}"

    except Exception as e:
        return f"Wiki lookup failed: {e}"


if __name__ == "__main__":
    print(search_wiki.invoke("Jon Snow")[:500])

NameError: name 'query' is not defined

In [28]:
class ResearchResponse(BaseModel):
  topic: str
  summary: str
  sources: list[str]
  tools_used: list[str]

parser = PydanticOutputParser(pydantic_object=ResearchResponse)

In [30]:
SYSTEM_PROMPT= """You answer questions about A Song of Ice and Fire.
You must call search_wiki before answering any factual question, every time,
even when you believe you know the answer. Base your answer only on what
search_wiki returns. If it returns nothing relevant, say the wiki does not
cover it. Always name the wiki page you used."""

In [31]:
agent = create_react_agent(llm, [search_wiki], prompt=SYSTEM_PROMPT)

NameError: name 'search_wiki' is not defined